# KíLín - Code Fixture Evaluation Notebook

This notebook asks a real model questions about the small mock repository in
`tests/fixtures/code_repo`, which has one file per supported code language.
It checks the whole path a question takes: routing, retrieval, the code
graph and the answer. How each file is read and split is tested without
models by `tests/kilin/test__code_fixture_repo.py`; this notebook checks
what a user would see.

Each pull request that teaches KíLín a new file type adds one file to the
fixture and a few questions to `QUESTIONS` below. A check is a hint for
review, not a verdict: read each answer.

## 1. Configuration

Models must already be installed locally when `ALLOW_DOWNLOAD=False`.

In [ ]:
from datetime import datetime
from pathlib import Path

EXPERIMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PROJECT_NAME = f"trial_code_fixture_{EXPERIMENT_ID}"
ALLOW_DOWNLOAD = False
BACKEND = "mlx"
CHAT_MODEL = "Qwen/Qwen3-8B-MLX-4bit"
GENERATION_SETTINGS = {
    "temperature": 0.0,
    "thinking": True,
    "max_new_tokens": 4096,
}

repo_root = Path.cwd().resolve()
while not (repo_root / "tests" / "fixtures" / "code_repo").is_dir():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run inside the KíLín development repository.")
    repo_root = repo_root.parent
fixture_dir = repo_root / "tests" / "fixtures" / "code_repo"
project_dir = repo_root / "sandbox" / PROJECT_NAME
print("Fixture:", fixture_dir)
print("Project:", project_dir)

## 2. Questions

Names in the fixture say what they are (`js_function_A` calls
`js_function_B`), so each answer can be checked by eye. Each question lists
the files its evidence should come from and the names the answer should
contain. `graph` marks questions whose answer needs the code graph, such as
callers that are not mentioned near the function itself.

In [ ]:
QUESTIONS = (
    {
        "id": "Q01",
        "question": "Which source files are in this project?",
        "paths": (),
        "terms": (
            "calculator_py.py", "calculator_js.js", "m_function_B.m",
            "page_html.html",
        ),
    },
    {
        "id": "Q02",
        "question": "Which function does js_function_A call?",
        "paths": ("js/calculator_js.js",),
        "terms": ("js_function_B",),
    },
    {
        "id": "Q03",
        "question": "Which class does JsChildClass extend, and what "
                    "does that class store?",
        "paths": ("js/calculator_js.js",),
        "terms": ("JsBaseClass", "value"),
    },
    {
        "id": "Q04",
        "question": "Which JavaScript function doubles a number?",
        "paths": ("js/calculator_js.js",),
        "terms": ("js_function_B",),
    },
    {
        "id": "Q05",
        "question": "Which functions or methods call js_function_A?",
        "paths": ("js/calculator_js.js",),
        "terms": ("js_method_C",),
        "graph": True,
    },
    {
        "id": "Q06",
        "question": "Which function does py_function_A call?",
        "paths": ("python/calculator_py.py",),
        "terms": ("py_function_B",),
    },
    {
        "id": "Q07",
        "question": "If I change py_function_B, which functions are "
                    "affected?",
        "paths": ("python/calculator_py.py",),
        "terms": ("py_function_A", "py_method_C"),
        "graph": True,
    },
    {
        "id": "Q08",
        "question": "Which function does m_function_A call to get the sum?",
        "paths": ("matlab/m_function_A.m",),
        "terms": ("m_function_B",),
    },
    {
        "id": "Q09",
        "question": "Which JavaScript file does page_html.html load?",
        "paths": ("html/page_html.html",),
        "terms": ("calculator_js.js",),
    },
    {
        "id": "Q10",
        "question": "Which function does the html_button_D button in "
                    "page_html.html call?",
        "paths": ("html/page_html.html",),
        "terms": ("js_function_A",),
    },
    {
        "id": "Q11",
        "question": "Which page uses js_function_A?",
        "paths": ("html/page_html.html", "js/calculator_js.js"),
        "terms": ("page_html.html",),
        "graph": True,
    },
)
print(len(QUESTIONS), "questions")

## 3. Create the Code project

The project uses the packaged Code type unchanged, so its file types and
`code_graph: true` are the ones users get.

In [ ]:
from kilin.workspace import add_project_sources, prepare_sources_workspace

if project_dir.exists():
    raise FileExistsError(f"Project already exists: {project_dir}")
project = prepare_sources_workspace(
    [], "code", project_dir,
    settings={
        "backend": BACKEND,
        "chat_mlx": CHAT_MODEL,
        "allow_download": ALLOW_DOWNLOAD,
        **GENERATION_SETTINGS,
    },
)
project = add_project_sources(project_dir, [fixture_dir])
print("File types:", project.profile.extensions)
print(
    "Copied files:",
    sorted(
        path.relative_to(project.source).as_posix()
        for path in project.source.rglob("*") if path.is_file()
    ),
)

## 4. Ask the questions

One conversation holds all questions, as a user would ask them. Opening
the session indexes the project and builds the code graph.

In [ ]:
from kilin.conversations import ConversationStore
from kilin.rag import RAGSession

store = ConversationStore(project_dir)
info = store.create()
session = RAGSession(project_dir, conversation_path=store.state_path(info))
session.use_chat_model(CHAT_MODEL)
session.chat_session.configure_generation(GENERATION_SETTINGS)

RESULTS = []
for case in QUESTIONS:
    answer = session.ask(
        case["question"], generation_settings=dict(GENERATION_SETTINGS)
    )
    turn = session.turns[-1]
    paths = []
    for item in turn.evidence:
        path = Path(item.path).as_posix()
        # Sources are copied under the fixture's folder name.
        path = path.removeprefix(f"{fixture_dir.name}/")
        if path not in paths:
            paths.append(path)
    text = answer.split("</think>")[-1]
    RESULTS.append({
        **case,
        "answer": text.strip(),
        "intent": turn.parameters.get("intent"),
        "evidence": paths,
        "missing_paths": [p for p in case["paths"] if p not in paths],
        "missing_terms": [t for t in case["terms"] if t not in text],
    })
    print(case["id"], "done")

## 5. Results

`OK` means every expected file was in the evidence and every expected word
is in the answer. `CHECK` lists what is missing; read the answer before
deciding whether it is wrong.

In [ ]:
for result in RESULTS:
    problems = result["missing_paths"] + result["missing_terms"]
    status = "OK" if not problems else "CHECK " + ", ".join(problems)
    graph = " [graph]" if result.get("graph") else ""
    print(f"=== {result['id']}{graph}: {status}")
    print("Question:", result["question"])
    print("Intent:  ", result["intent"])
    print("Evidence:", ", ".join(result["evidence"]) or "-")
    print(result["answer"])
    print()

## 6. Close

Close the session to release the models. The project stays in `sandbox/`
for inspection in the app.

In [ ]:
session.close()
print("Project kept at:", project_dir)